In [ ]:
import os
import sys

import tensorflow as tf
import yaml

sys.path.append(os.path.abspath("../"))
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import glob

import matplotlib.pyplot as plt
import numpy as np
import sklearn

from train.models import FullModel
from util import dataset as u_dataset
from util import keypoint as u_keypoint

In [ ]:
batch_size = 1840

model_name = "20251030-154803.keras"
path_to_models = "../../models/finished/"

In [ ]:
# Get encoder config
def load_config(config_path):
    with open(config_path) as f:
        config = yaml.safe_load(f)
    return config


config = load_config(f"../../logs/fit/{model_name.split('.')[0]}/config.yaml")

encoder_architecture = config["model"]["encoder"]["architecture"]
classifier_architecture = config["model"]["classifier"]["architecture"]
categories_config = config["categories"]

#### Load Test Data


In [ ]:
path_to_test_data = glob.glob("../../data/tfrecords/test_ds*.tfrecords")

test_ds = u_dataset.get_dataset(path_to_test_data)
test_ds = test_ds.batch(batch_size, drop_remainder=False)

#### Load Model

In [ ]:
model = FullModel.load(
    encoder_architecture,
    classifier_architecture,
    input_dims=config["model"]["encoder"]["input_dims"],
    filepath=path_to_models,
    filename=model_name,
    n_context=config["model"]["encoder"]["n_context"],
    only_train_encoder=config["model"]["encoder"]["only_train_encoder"],
    classifier_offsets=config["model"]["classifier"]["with_offsets"],
    encoder_only=False,
    verbose=True,
    n_meta=config["model"]["classifier"]["n_meta"],
    encoder_use_batch_norm=config["model"]["encoder"]["use_batch_norm"],
    classifier_use_batch_norm=config["model"]["classifier"]["use_batch_norm"],
    categories_config=config["categories"],
)
model.compile(optimizer=tf.keras.optimizers.Adam(), jit_compile=False)


#### Predict Data

In [ ]:
groundtruth_dataset = []
for x in test_ds:
    groundtruth_dataset.append(x)

# Take only image, camera, intrinsics for input
input_data = test_ds.map(lambda x: (x["image"], x["camera"], x["intrinsics"]))

# Convert tf.Data.Dataset into list
input_list = []
for x, y, z in input_data:
    input_list.append((x, y, z))

model.run_eagerly = True

predictions = [model.predict(x=batch) for batch in input_list]
metrics_list = [model.evaluate(x=batch, return_dict=True) for batch in groundtruth_dataset]

#### Evaluate Models

In [ ]:
# Calculate mean for metrics over each batch
keys = metrics_list[0].keys()
stacked_values = {key: tf.stack([metrics[key] for metrics in metrics_list]) for key in keys}
metrics_mean = {key: tf.reduce_mean(stacked_values[key]) for key in keys}

# Print Metrics
for key, value in metrics_mean.items():
    print(f"{key}: {value.numpy():.4f}")

### Accuracy, Precision, Recall

In [ ]:
def reshape_tensors(object_tensors, reshape=True):
    # object_tensors = [tensor[object_name] for tensor in tensors]

    stacked_tensors = {
        key: tf.stack([object_tensor[key] for object_tensor in object_tensors])
        for key in object_tensors[0]
    }
    if reshape:
        return {
            key: tf.reshape(tensor, (-1, *tensor.shape[2:]))
            for key, tensor in stacked_tensors.items()
        }
    else:
        return stacked_tensors

In [ ]:
def get_conf_matrix(predictions, groundtruth, encoder_threshold, classifier_threshold):
    """Calculate y_pred. A binary tensor which is True if an object was detected in the sample and False if no object was detected.

    A prediction counts as positive if:
        - the combined prediction (encoder prediction + classifier prediction) is greater-equal than the combined threshold (encoder_threshold + classifier_threshold)
        - The predicted patch coordinates could be projected on the ground
        - If the predicted patch is actually over the object.

    Args:
        predictions: The model predictions.
        groundtruth: The corresponding groundtruth data.
        encoder_threshold: The threshold of the encoder.
        classifier_threshold: The threshold of the classifier.

    Returns:
        dict() containing fp, tp, fn, tn
    """

    # Get the encoder logits where a patch was drawn (those are the one with the highest predicted probability)
    best_logits = tf.gather(
        predictions["logits"], predictions["patch_indices"], batch_dims=1
    )  # (B, 5)

    # Add the encoder prediction and the classifier prediction to a combined prediction
    combined_predictions = best_logits + predictions["classification"]  # (B, 5)

    # The groundtruth coordinates of the object
    coords_true = u_keypoint.get_coords_from_offsets(groundtruth["offset_mask"])  # (B, 2)
    object_in_image = tf.math.reduce_any(
        tf.cast(groundtruth["object_mask"], tf.bool), axis=[1, 2]
    )  # (B, )

    # The patch_indices with the best combined prediction score
    best_score_index = tf.argmax(combined_predictions, axis=-1)  # (B, )

    # The best prediction of each sample.
    best_predictions = tf.gather(combined_predictions, best_score_index, batch_dims=1)  # (B, )

    # The best box of each sample
    best_boxes = tf.gather(predictions["boxes"], best_score_index, batch_dims=1)  # (B, 4)

    coords_true_normalized = coords_true / [640, 480]  # [B, 2]
    # Is True if the best predicted box is actually on the object.
    valid_boxes = u_keypoint.are_coords_in_patch(coords_true_normalized, best_boxes)  # (B, )
    # print(best_boxes)

    # Is True if the prediction of the best patch is greater-equal the combined threshold
    over_threshold = best_predictions >= (encoder_threshold + classifier_threshold)  # (B, )

    fp = over_threshold & tf.math.logical_not(valid_boxes)
    tp = over_threshold & valid_boxes
    fn = tf.math.logical_not(over_threshold) & object_in_image
    tn = tf.math.logical_not(over_threshold) & tf.logical_not(object_in_image)

    return {
        "fp": tf.reduce_sum(tf.cast(fp, tf.int32)),
        "tp": tf.reduce_sum(tf.cast(tp, tf.int32)),
        "fn": tf.reduce_sum(tf.cast(fn, tf.int32)),
        "tn": tf.reduce_sum(tf.cast(tn, tf.int32)),
    }


In [ ]:
groundtruth_reshaped = reshape_tensors([batch["ball"] for batch in groundtruth_dataset], True)
ball_predictions_reshaped = reshape_tensors(
    [batch["results"]["ball"] for batch in predictions], True
)
thresholds = np.linspace(0, 1, 40, endpoint=True)


conf_values_cla = [
    get_conf_matrix(ball_predictions_reshaped, groundtruth_reshaped, 0, cla) for cla in thresholds
]
# conf_values_enc = [
#     get_conf_matrix(ball_predictions_reshaped, groundtruth_reshaped, enc, 0) for enc in thresholds
# ]

In [ ]:
classifier_threshold = 0.99
encoder_threshold = 0

In [ ]:
precision_cla = [(x["tp"] / (x["tp"] + x["fp"])).numpy() for x in conf_values_cla]
recall_cla = [(x["tp"] / (x["tp"] + x["fn"])).numpy() for x in conf_values_cla]

plt.plot(thresholds, precision_cla, label="Precision")
plt.plot(thresholds, recall_cla, label="Recall")
plt.title("Precision/Recall Curve for Classifier Threshold")
plt.axvline(classifier_threshold, ls="--", color="r")
plt.xlabel("Threshold")
plt.ylabel("Precision/Recall")
plt.legend()
plt.show()

In [ ]:
a = get_conf_matrix(
    ball_predictions_reshaped, groundtruth_reshaped, encoder_threshold, classifier_threshold
)

In [ ]:
tf.print("Precision: ", a["tp"] / (a["tp"] + a["fp"]))
tf.print("Recall: ", a["tp"] / (a["tp"] + a["fn"]))
tf.print("fp rate:", a["fp"] / (a["fp"] + a["tp"]))
tf.print("fn rate: ", a["fn"] / (a["fn"] + a["tn"]))

sklearn.metrics.ConfusionMatrixDisplay(np.array([[a["tn"], a["fp"]], [a["fn"], a["tp"]]])).plot()